In [4]:
%load_ext autoreload
%autoreload 2

from notebook_functions import criar_tabuleiro_vazio, simular_stack_e_pop
from main import main_menu_text, select_ai_menu_text, play_menu_text
from main import display_rules
from ui import UI

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## PopOut: Adversarial Search Strategies and Decision Trees

Alunos: Aly Mohamed (20240145), Rafael (xxxx), Victor (xxxx)

### 1 - Implementação do jogo em Python.


Foram utilizados 03 ficheiros para viabilizar a implementação de toda a estrutura do jogo, board.py, ui.py, main.py

##### 1.1 board.py

A classe Board atua como o principal engine do jogo, sendo responsável por gerir toda a lógica, regras e o estado atual da partida. No seu núcleo, a classe é inicializada e guarda dois elementos fundamentais na memória: a GRID e o CURRENT_PLAYER

Para viabilizar a integração com os algoritmos de Inteligência Artificial, as funcionalidades da classe estão divididas em três eixos principais:

- Movimento. Feito pelas funções drop_piece e is_valid_drop, a qual identifica se existe um espaço livre na coluna desejada e, caso sim, insere a peça do atual jogador lá. Também temos a função pop_piece e is_valid_pop, que verifica se a base da coluna possui uma peça igual a do jogador da vez e, caso sim, remove-a fazendo a translação de todas as peças acima para 1 posiçaõ abaixo.

- Checks de estados terminais. Feito pelas funções check-win, que varre o tabuleiro nas quatro direções e retorna true caso haja alinhamento de 4 peças iguais, e a função is_full que avalia o topo de cada coluna e retorna positivo caso não haja mais espaços disponiveis.

- Integração (IA). Só possível pelas funções get_legal_moves, que analisa o tabuleiro e devolve uma lista formatada com todas as ações válidas no momento, copy, que executa uma cópia profunda do objeto atual sem perde-lo em meio a iterações, e get_state, que converte o grid em tupla de tuplas permitindo eficiência de memória durante o treino dos modelos.




#### 1.2 ui.py

A classe UI (User Interface) foi desenhada com métodos estáticos para gerir exclusivamente a representação visual do jogo e a interação no terminal. As funcionalidades desta classe estão divididas em duas componentes principais:

- Renderização de Estado: feito pelo método render, em que sua chamada é responsável por limpar o ecrã, imprimir a mensagem de contexto(aviso de turnos ou vitória) e imprimir o tabuleiro.

In [7]:
tabuleiro_demo = criar_tabuleiro_vazio()
UI.print_board(tabuleiro_demo)


  1   2   3   4   5   6   7
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|



- Animação visual: utilizando do método copy da classe Board, ele gera estados temporários. E através desses estados temporários, é possível dar a ilusão ao utilizador humano de "lançamento" de uma peça através dos métodos(classe UI) animated_drop e animated_pop.

In [8]:
simular_stack_e_pop(tabuleiro_demo, coluna=3, num_pecas=4)

Demonstração finalizada! A coluna está totalmente limpa.

  1   2   3   4   5   6   7
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|



#### 1.3 main.py

É nesse ficheiro que ocorre a integração entre a criação do jogo e suas regras(board.py) e visualiações (ui.py), com o usuário(terminal) e algoritmos de inteligência artificial (IA). Para isso, foi desenvolvida uma interface gráfica iterativa que permite que o usuário navegue pelo código para selecionar a opção que melhor deseja.

##### 1.3.1 Interface - main.py

In [2]:
main_menu_text()

  Welcome to PopOut on terminal!   
 1 - Play
 2 - Rules
 3 - Credits
 4 - Exit Game


In [5]:
play_menu_text()

           SELECT MODE             
 1 - Human Vs Human
 2 - Human vs AI
 3 - AI vs AI
 4 - Back


In [6]:
select_ai_menu_text()


 Select Algorithm for AI Player 
 1 - MCTS Heuristic (With heuristics and optimizations)
 2 - MCTS Vanilla (Standard)
 3 - MCTS Multi-Expansion (N-Children)
 4 - Decision Tree


##### 1.3.2 Jogabilidade - main.py

Toda a jogabilidade é gerida a partir da função play_game(). Em ordem de execução, essa função é responsável por:
- criar o objeto da classe Board.
- criar um dicionário vázio (responsável por guardar os estados do jogo para contabilizar a regra de empate), ao mesmo tempo já adiciona o estado inicial com a contagem de 1.
- Busca as variáveis globais que representam os tipos de jogadores (Humanos ou IA's).
- Inicializa o Loop de jogadas que irá ser repetido até a flag game_over seja true.

Quanto ao 